# Homework

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install langchain openai langchain_experimental langchain_community langchain_ollama ollama faiss-cpu huggingface_hub[hf_xet]
!pip install streamlit streamlit_chat

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.3 MB/s eta 0:00:00


## Learning for how to write a simple RAG!

In [ ]:
from langchain_community.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import CSVLoader
from langchain_ollama import OllamaEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)
from langchain import LLMChain
from langchain.prompts import PromptTemplate

import openai
import time
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import os
import pandas as pd
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
#API key
user_api_key="sk-c8f499bd18954c3097f072c14ba1eb25"     #Your personal Deepseek API key.
langchain_api_key = os.getenv('LANGCHAIN_API_KEY')
langchain_tracing_v2 = os.getenv('LANGCHAIN_TRACING_V2', 'true')
langchain_endpoint = os.getenv('LANGCHAIN_ENDPOINT', 'https://api.smith.langchain.com')
#LangSmith for tracking
os.environ['LANGCHAIN_TRACING_V2'] = langchain_tracing_v2
os.environ['LANGCHAIN_ENDPOINT'] = langchain_endpoint
os.environ['LANGCHAIN_API_KEY'] = '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'

In [ ]:
query="""

My question about the content from csv file or the txt file.

"""


# Initialize the LLM. If you just need to retrieve the relevant document without further generation, this step no need.
llm = ChatOpenAI(
    model="deepseek-chat",
    # model="deepseek-reasoner",
    openai_api_key=user_api_key,
    openai_api_base="https://api.deepseek.com",
    temperature=0.0
)

# Load and process the data
CSVloader = CSVLoader(file_path="/content/drive/MyDrive/Colab Notebooks/Data/All_Diets.csv", encoding="utf-8")
#YOU can also retrieve any text document you prefer and further asking the RAG with details among the contents.
# TEXTloader = TextLoader(file_path="WHICHEVER CSV DOCUMENT YOU PREFER.txt", encoding="utf-8")

data = CSVloader.load()

# Each line is a document
documents = data

# Create embeddings and vector store
# embeddings = OllamaEmbeddings(model="llama3")
# vectors = FAISS.from_documents(documents, embeddings)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors = FAISS.from_documents(documents, embeddings)

# Create a retriever, with tok K most similar documents retrieved from the csv or txt file.
retriever = vectors.as_retriever(search_kwargs={'k': 2})


#Now you can simply get the top K most similar document
similar_docs = retriever.get_relevant_documents(query)



# Or if you go further for the generation from the whole RAG, create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)
#RA+Generation
answer=qa_chain.invoke(query)
#the output will have three parts. Query is the query that you ask. Result is the answer generated by llm after retrieving the relevant k documents. Source_documents are the k documents retrieved.
print(answer['result'])

# Create a tool using the RetrievalQA chain
qa_tool = Tool(
    name="ORLM_QA",
    func=qa_chain.invoke,
    description=(
        "Use this tool to answer questions."
        "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
        #"In the content of the file, content in label is generated taking 'text' and 'information' into account at the same time."
    ),
)


#create an agent for final
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#     agent_kwargs={
#         "prefix": prefix.format(few_shot_examples=few_shot_examples),
#         "suffix": suffix,
#     },
    verbose=True,
    handle_parsing_errors=True,  # Enable error handling
)

response = agent.invoke(query)


<ipython-input-24-86408d3277e1>:38: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  similar_docs = retriever.get_relevant_documents(query)


Based on the provided context, I can see two recipes with their details:

1. **Vegan Coconut-Lime Panna Cotta with Mango Jam**
   - Cuisine type: American  
   - Protein: 17.71g  
   - Carbs: 157.36g  
   - Fat: 54.27g  

2. **Tomatoes Stuffed with Puréed Cauliflower and Feta**
   - Cuisine type: Mediterranean  
   - Protein: 44.16g  
   - Carbs: 57.12g  
   - Fat: 80.85g  

If you have a specific question about these recipes or the data, feel free to ask! If you're looking for something beyond this context, I may not have the answer.


> Entering new AgentExecutor chain...


<ipython-input-24-86408d3277e1>:67: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


Since the question is about content from a CSV or TXT file, I should use the `ORLM_QA` tool to retrieve and answer the question. However, the question itself is not specified. I need the user to provide the actual question they want answered about the file's content.

Final Answer: Could you please provide the specific question you'd like answered about the content of the CSV or TXT file? For example, "What is the total number of rows in the CSV file?" or "What does the third line of the TXT file say?" Once you provide the question, I can retrieve the relevant information for you.

> Finished chain.


### Question 1

**Now you have a recipe of an awesome restaurant that can cook more than 7000 dishes with cuisine type of American, Asian, British, Caribbean, Central Europe, French, Indian, Italian, Japanese, Kosher, Mediterranean, Mexican, Middle Eastern, Nordic, South American, South East Asian. You also have the nutrition detail for every dish (protein, carbs, fats). Your job is to prepare a QnA system (hint. through the output of the Agent-invoke), that can help to figure out food combination that is suiable for a man's daily consumption.**

For example:
"If I am a Chinese cuisine guy, what two dishes are you recommand that will just barely cover my consumption of 200 grams of protein and 300 grams of Carbs?"

In [ ]:
import streamlit as st
from streamlit_chat import message
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.vectorstores import FAISS
import tempfile
from langchain_experimental.agents.agent_toolkits import create_csv_agent
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
import pandas as pd
from langchain.chains import RetrievalQA
from langchain.agents import initialize_agent, AgentType, Tool
import re
from datetime import datetime, time
from langchain_community.document_loaders import TextLoader

from langchain import LLMChain
from langchain.prompts import PromptTemplate

import openai
import time
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import os
import pandas as pd

In [ ]:
user_api_key='sk-67de119475eb4d73b91fd5a6049d71d9'

In [ ]:
llm = ChatOpenAI(
    model="deepseek-chat",
    # model="deepseek-reasoner",
    openai_api_key=user_api_key,
    openai_api_base="https://api.deepseek.com",
    temperature=0.0
)

# Load and process the data
loader = CSVLoader(file_path="/content/drive/MyDrive/Colab Notebooks/Data/All_Diets.csv", encoding="utf-8")
data = loader.load()

# Each line is a document
documents = data

# Create embeddings and vector store
# embeddings = OllamaEmbeddings(model="llama3")
# vectors = FAISS.from_documents(documents, embeddings)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors = FAISS.from_documents(documents, embeddings)

# Create a retriever
retriever = vectors.as_retriever()

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

# Create a tool using the RetrievalQA chain
qa_tool = Tool(
    name="FileQA",
    func=qa_chain.invoke,
    description=(
        "Use this tool to answer questions about the problem type of the text. "
        "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
    ),
)

added module:-------------------------

In [ ]:
TEXTloader = TextLoader(file_path="/content/drive/MyDrive/Colab Notebooks/Data/vegfood.txt", encoding="utf-8")
textdata = TEXTloader.load()
embeddings1 = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors1 = FAISS.from_documents(textdata, embeddings1)
retriever1 = vectors1.as_retriever()
textref_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever1,
    return_source_documents=True,
)
textref_tool = Tool(
    name="TextRef",
    func=textref_chain.invoke,
    description=(
        "Use this tool to find useful information about the extra requirement of the text. "
        "Provide the generated question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
    ),
)

-------------------------------:added module

In [ ]:
# Create the prefix and suffix for the agent's prompt
# prefix = """You are a helpful assistant that can answer questions about uploading txt file. When giving the final answer, always answer in the following format:

# "dishes name, actual nutrients amount of these dishes"

# Always use the FileQA tool when you need to retrieve information from the file. When you need to find information from the file, use the provided tools.

# """
prefix = """You are a nutritional expert assistant. You must follow these steps:
1. Identify user's cuisine preference, and the number of dishes the user need
2. Determine nutritional targets (protein, carbs)
3. Generate an input question in the form of:
The user needs the cusine type, the number of dishes is,
the target protein is, the target carbons is,
the extra requirement is, please generate dishes combination that meet the requirements.
4. Use the FileQA tool to answer the question generated.
5. If no information can be retrieved from FileQA tool and there are extra requirements, consult the TextRef tool for useful information.
"""
suffix = """Generate the final output in the following format:
【Recommended Dishes】
1. Dish A -  protein, carbs
2. Dish B -  protein, carbs
【Total】Protein: g | Carbs: g
"""
agent = initialize_agent(
    tools=[qa_tool, textref_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    agent_kwargs={
        "prefix": prefix,
        "suffix": suffix,
    },
)

In [ ]:
result=agent.invoke('If I am a Chinese cousine guy and a vegetarian, what two dishes are you recommand that with each serving will just barely cover my consumption of 200 grams of protein and 300 grams of Carbs? If two dishes are not enough, I can accept 3 dishes.')



> Entering new AgentExecutor chain...
Question: The user needs the cuisine type Chinese, the number of dishes is 2-3, the target protein is 200g, the target carbohydrates is 300g, the extra requirement is vegetarian, please generate dishes combination that meet the requirements.

Thought: I need to find Chinese vegetarian dishes that can meet the high protein and carbohydrate requirements. Since the user is open to 3 dishes if 2 are insufficient, I'll first check for 2-dish combinations.

Action:
```
{
  "action": "FileQA",
  "action_input": "The user needs the cuisine type Chinese, the number of dishes is 2-3, the target protein is 200g, the target carbohydrates is 300g, the extra requirement is vegetarian, please generate dishes combination that meet the requirements."
}
```

Observation: {'query': 'The user needs the cuisine type Chinese, the number of dishes is 2-3, the target protein is 200g, the target carbohydrates is 300g, the extra requirement is vegetarian, please generate 

In [ ]:
print(result['output'])

Based on the information gathered, here's a combination of three vegetarian Chinese dishes that meet your nutritional targets:

【Recommended Dishes】
1. Vegetarian Mapo Tofu - 100g protein, 30g carbs
2. Stir-Fried Eggplant with Garlic Sauce - 30g protein, 70g carbs
3. Vegetarian Fried Rice with Mixed Vegetables & Tofu - 70g protein, 200g carbs

【Total】Protein: 200g | Carbs: 300g

This combination provides:
- High-protein tofu dishes (Mapo Tofu and Fried Rice)
- Carbohydrate-rich fried rice to meet your carb target
- Balanced flavors and textures typical of Chinese cuisine
- All vegetarian ingredients while meeting your strict nutritional requirements

The portions can be adjusted slightly if you need to fine-tune the exact protein/carb counts. Would you like any modifications to this meal plan?


### !!! 附加题

**Yet this combination meet the demand of your requirements. Do you think this combination is healthy enough? How can you fix this problem if I am a vegiterian? Or I'd like one vegetarian dish and one meat dish?**

Also think of traditional knapsack problem.

Suppose every customer who comes to consume has three states: I want to lose weight, I want to shape my body, and I want to build muscle.

Corresponding to these states, the nutrient content should be 70% - 90% (lose weight), 90% - 110% (shape my body), and the protein intake should be no less than 150% of the daily requirement, while other nutrients should not exceed 100% of the daily requirement (build muscle).
Please help me design a system that, after each customer provides their daily nutrient needs and their state, offers them a two-course meal set and a three-course meal set respectively, ensuring that the overall nutrient content of the meal sets meets the requirements and the meat and vegetable combination is reasonable.

(Hint: Maybe try to use prompt engineering (few shot example) in the agent setting?)

In [ ]:
import streamlit as st
from streamlit_chat import message
from langchain_community.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import CSVLoader
from langchain_ollama import OllamaEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)
from langchain import LLMChain
from langchain.prompts import PromptTemplate

import openai
import time
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import os
import pandas as pd
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
user_api_key='sk-67de119475eb4d73b91fd5a6049d71d9'

In [ ]:
llm = ChatOpenAI(
    model="deepseek-chat",
    # model="deepseek-reasoner",
    openai_api_key=user_api_key,
    openai_api_base="https://api.deepseek.com",
    temperature=0.0
)

# Load and process the data
loader = CSVLoader(file_path="/content/drive/MyDrive/Colab Notebooks/Data/All_Diets.csv", encoding="utf-8")
data = loader.load()

# Each line is a document
documents = data

# Create embeddings and vector store
# embeddings = OllamaEmbeddings(model="llama3")
# vectors = FAISS.from_documents(documents, embeddings)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors = FAISS.from_documents(documents, embeddings)

# Create a retriever
retriever = vectors.as_retriever()

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

# Create a tool using the RetrievalQA chain
qa_tool = Tool(
    name="FileQA",
    func=qa_chain.invoke,
    description=(
        "Use this tool to answer questions about the problem type of the text. "
        "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
    ),
)

<ipython-input-6-9d976a64c6c5>:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or d

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
df = pd.DataFrame([doc.page_content for doc in documents])

import pandas as pd

columns = ['Recipe_name', 'Cuisine_type', 'Protein', 'Carbs', 'Fat']
new_df = pd.DataFrame(columns=columns)

for index, row in df.iterrows():
    data = row[0].split('\n')

    new_row = {}

    for col, item in zip(columns, data):
        key_value = item.split(':')

        key = key_value[0].strip()
        value = key_value[1].strip()

        new_row[col] = value

    new_df = pd.concat([new_df, pd.DataFrame([new_row])], ignore_index=True)



new_df[['Protein','Carbs','Fat']] = new_df[['Protein','Carbs','Fat']].apply(pd.to_numeric)
dtype_dict = {
    'Recipe_name': 'category',
    'Cuisine_type': 'category',
    'Protein': 'float32',
    'Carbs': 'float32',
    'Fat': 'float32'
}
new_df = new_df.astype(dtype_dict)
df = new_df.copy()

In [ ]:
from langchain.tools import tool
from typing import Tuple
from pydantic import BaseModel, Field

class NutritionRequirement(BaseModel):
    state: str = None
    protein_range: Tuple[float, float] = [50,300]
    carbs_range: Tuple[float, float] = [45,300]
    fats_range: Tuple[float, float] = [10,60]

    def __init__(self, state, base_protein, base_carbs, base_fats, **data):
        super().__init__(**data)
        self.state = state  # lose_weight/shape/build_muscle
        self._adjust_targets(base_protein, base_carbs, base_fats)

    def _adjust_targets(self, p, c, f):
        if self.state == "lose_weight":
            self.protein_range = (p*0.7, p*0.9)
            self.carbs_range = (c*0.7, c*0.9)
            self.fats_range = (f*0.7, f*0.9)
        elif self.state == "shape":
            self.protein_range = (p*0.9, p*1.1)
            self.carbs_range = (c*0.9, c*1.1)
            self.fats_range = (f*0.9, f*1.1)
        elif self.state == "build_muscle":
            self.protein_range = (p*1.5, float('inf'))
            self.carbs_range = (0, c)
            self.fats_range = (0, f)

@tool
def get_nutrition_advice(state: str, base_protein: float, base_carbs: float, base_fats: float):
    """
    This tool is used to provide nutrition advice based on the user's state, base protein, base carbs, and base fats.
    It returns a NutritionRequirement object.
    """
    # Call NutritionRequirement with keyword arguments
    return NutritionRequirement(state=state, base_protein=base_protein, base_carbs=base_carbs, base_fats=base_fats)

In [ ]:
TEXTloader = TextLoader(file_path="/content/drive/MyDrive/Colab Notebooks/Data/healthy_diet.txt", encoding="utf-8")
textdata = TEXTloader.load()
embeddings1 = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors1 = FAISS.from_documents(textdata, embeddings1)
retriever1 = vectors1.as_retriever()
textref_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever1,
    return_source_documents=True,
)
textref_tool = Tool(
    name="TextRef",
    func=textref_chain.invoke,
    description=(
        "Use this tool to find useful information about the health dimension of the text. "
        "Provide the generated question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
    ),
)

In [ ]:
prefix = """As a nutrition expert, you must follow these steps:
1. Identify the user's state: {state}
2. Identify the user's daily nutrient requirements: Protein{protein}g Carbs{carbs}g Fat{fats}g
3. Use get_nutrition_advice tool to enerate adjuested nutrient goal based on the user's state
4. Search the dishes that match the cuisine preference
5. Use qa_tool to generate a combination of 2 dishes and a combination of 3 dishes that the total nutrition of each \
combination both meet the nutrient goal and that match the cuisine preference
6. If the retrieved results don't meet the requirement, consult the TextRef tool for useful information.
7. Guarantee the balance between meat and vegetable"""

suffix = """Generate the final output in the following format:
【Recommendation for {state}】
- Two dishes：
{combo2}
- Three dishesL
{combo3}

Total nutrition meet:
Protein：{p_range}
Carbs：{c_range}
Fat：{f_range}
"""


agent = initialize_agent(
    tools=[get_nutrition_advice, qa_tool, textref_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    agent_kwargs={
        "prefix": prefix,
        "suffix": suffix,
    },
    verbose=True,
)

In [ ]:
state = 'lose_weight'
base_p = 90
base_c = 150
base_f = 60
cuisine = 'Chinese'

# Provide initial values for the missing keys
query = f"I want to {state}, my requirement on daily protein is {base_p}g, daily carbon is {base_c}, \
        daily fat is {base_f} and I hope you can recommend {cuisine} type of 2 dishes combination \
        and 3 dishes combination for me. Besides, \
        please list the total proteins and carbs and fats of your recommended combinations."

result = agent.run(
    {
        "state": state,
        "protein": base_p,
        "carbs": base_c,
        "fats": base_f,
        "cuisine": cuisine,
        "input": query,
        "combo2": "",
        "combo3": "",
        "p_range": "",
        "c_range": "",
        "f_range": "",
    }
)



> Entering new AgentExecutor chain...
Thought: The user wants to lose weight and has specified their daily nutrient requirements. They prefer Chinese cuisine and want combinations of 2 and 3 dishes that meet their nutrient goals. I will first adjust the nutrient goals based on their weight loss state, then search for suitable Chinese dishes, and finally provide balanced combinations.

Action:
```
{
  "action": "get_nutrition_advice",
  "action_input": {
    "state": "lose_weight",
    "base_protein": 90,
    "base_carbs": 150,
    "base_fats": 60
  }
}
```
Observation: state='lose_weight' protein_range=(62.99999999999999, 81.0) carbs_range=(105.0, 135.0) fats_range=(42.0, 54.0)
Thought:```
{
  "action": "FileQA",
  "action_input": "What are some healthy Chinese dishes suitable for weight loss that meet protein 63-81g, carbs 105-135g, and fats 42-54g?"
}
```
Observation: {'query': 'What are some healthy Chinese dishes suitable for weight loss that meet protein 63-81g, carbs 105-135g, 

In [ ]:
print(result)

【Recommendation for lose_weight】
- Two dishes：
  1. **Steamed Fish with Ginger & Scallions** (150g fish)  
  2. **Stir-Fried Bok Choy with Garlic** (1 cup) + **Brown Rice** (½ cup cooked)  

- Three dishes：
  1. **Chicken & Broccoli Stir-Fry** (150g chicken, 1 cup broccoli, light oil)  
  2. **Quinoa** (½ cup cooked)  
  3. **Sautéed Mushrooms with a Dash of Sesame Oil**  

Total nutrition meet:
Protein：~70g (2 dishes) / ~75g (3 dishes)
Carbs：~120g (2 dishes) / ~110g (3 dishes)
Fat：~45g (2 dishes) / ~50g (3 dishes)
